# Implementing various types of Vector Space Models

## Step 1:- Loading the data

In [3]:
import json
import csv
import os

# Define file paths
corpus_path = "scifact/corpus.jsonl"
queries_path = "scifact/queries.jsonl"
qrels_train_path = "scifact/qrels/train.tsv"
qrels_test_path = "scifact/qrels/test.tsv"

# 1. Load the Corpus (All documents)
corpus = {}
with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        corpus[doc['_id']] = doc

# 2. Load the Queries (All search terms)
queries = {}
with open(queries_path, 'r', encoding='utf-8') as f:
    for line in f:
        query = json.loads(line)
        queries[query['_id']] = query['text']

# 3. Helper function to load a qrels file
def load_qrels(file_path):
    qrels_dict = {}
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found.")
        return qrels_dict
        
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        next(reader) # Skip the header row (query-id, corpus-id, score)
        for row in reader:
            query_id, corpus_id, score = row
            if query_id not in qrels_dict:
                qrels_dict[query_id] = {}
            qrels_dict[query_id][corpus_id] = int(score)
    return qrels_dict

# Load train and test qrels separately
qrels_train = load_qrels(qrels_train_path)
qrels_test = load_qrels(qrels_test_path)

# Combine them into a single master evaluation dictionary
qrels_all = {**qrels_train, **qrels_test}

# Print summary diagnostics
print(f"--- Dataset Statistics ---")
print(f"Total documents loaded in Corpus : {len(corpus)}")
print(f"Total queries loaded            : {len(queries)}")
print(f"Queries with TRAIN answer keys  : {len(qrels_train)}")
print(f"Queries with TEST answer keys   : {len(qrels_test)}")
print(f"Total queries with answer keys  : {len(qrels_all)}")

--- Dataset Statistics ---
Total documents loaded in Corpus : 5183
Total queries loaded            : 1109
Queries with TRAIN answer keys  : 809
Queries with TEST answer keys   : 300
Total queries with answer keys  : 1109


## Step 2:- Text Preprocessing

This step involves:
* **Lowercasing:** 
* **Removing Punctuation**
* **Tokenization:** 

In [4]:
import re

# 1. Define our text cleaning pipeline
def preprocess_text(text):
    if not text:
        return []
        
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    
    return tokens

# 2. Process the Corpus
tokenized_corpus = {}
for doc_id, doc in corpus.items():
    full_text = doc.get('title', '') + " " + doc.get('text', '')
    tokenized_corpus[doc_id] = preprocess_text(full_text)

# 3. Process the Queries
tokenized_queries = {}
for query_id, text in queries.items():
    tokenized_queries[query_id] = preprocess_text(text)

# 4. Verify the results
sample_query_id = list(queries.keys())[0]
print("--- Preprocessing Verification ---")
print("ORIGINAL QUERY:")
print(queries[sample_query_id])
print("\nTOKENIZED QUERY:")
print(tokenized_queries[sample_query_id])

--- Preprocessing Verification ---
ORIGINAL QUERY:
0-dimensional biomaterials lack inductive properties.

TOKENIZED QUERY:
['0dimensional', 'biomaterials', 'lack', 'inductive', 'properties']


- **Calculating TF-IDF Matrix**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Prepare corpus strings and track their IDs in an ordered list
corpus_ids = list(tokenized_corpus.keys())
corpus_strings = [" ".join(tokens) for tokens in tokenized_corpus.values()]

print("Step 1: Initializing and fitting TF-IDF Vectorizer...")

vectorizer = TfidfVectorizer()
X_corpus = vectorizer.fit_transform(corpus_strings)
vocab = vectorizer.vocabulary_
print(f"Done! Corpus matrix shape: {X_corpus.shape}")

Step 1: Initializing and fitting TF-IDF Vectorizer...
Done! Corpus matrix shape: (5183, 50625)


## Step 3:- Modeling

- Normal VSM model
- Performing VSM in LSI space
- VSM with Transformer Embeddings

### Common Function

- only the input space changes for these three models but the documents retrival is same. So, written a reusable function that can be used for Document Retrival using VSM.

# Vector Space Model (VSM)

In the Vector Space Model, both documents and queries are represented as vectors in a high-dimensional term space.

The similarity between a query \(Q\) and a document \(D\) is computed using Cosine Similarity:

$$
Sim(Q,D)
=
\frac{Q \cdot D}
{\|Q\| \, \|D\|}
$$

where:

- \(Q \cdot D\) is the dot product of the query and document vectors.
- \(\|Q\|\) is the magnitude of the query vector.
- \(\|D\|\) is the magnitude of the document vector.

The dot product is:

$$
Q \cdot D
=
\sum_{i=1}^{n}
q_i d_i
$$

The vector magnitudes are:

$$
\|Q\|
=
\sqrt{\sum_{i=1}^{n} q_i^2}
$$

$$
\|D\|
=
\sqrt{\sum_{i=1}^{n} d_i^2}
$$

In TF-IDF based VSM, each term weight is computed as:

$$
w_{t,d}
=
tf(t,d)\times idf(t)
$$

where:

$$
idf(t)
=
\log\left(\frac{N}{df(t)}\right)
$$

and:

- \(tf(t,d)\) = frequency of term \(t\) in document \(d\)
- \(df(t)\) = number of documents containing term \(t\)
- \(N\) = total number of documents in the collection

Documents are ranked in descending order of cosine similarity, with higher scores indicating greater relevance to the query.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_documents(
        query_vectors,
        document_vectors,
        corpus_ids,
        top_k=10
):
    similarity_matrix = cosine_similarity(
        query_vectors,
        document_vectors
    )

    results = {}
    for q_idx in range(similarity_matrix.shape[0]):
        scores = similarity_matrix[q_idx]
        ranked_indices = np.argsort(scores)[::-1]
        results[q_idx] = [
            (corpus_ids[idx], scores[idx])
            for idx in ranked_indices[:top_k]
        ]

    return results

### 1. Normal VSM

In [7]:
# Prepare query strings
query_ids = list(tokenized_queries.keys())

query_strings = [
    " ".join(tokens)
    for tokens in tokenized_queries.values()
]

# Transform queries into TF-IDF space
X_queries = vectorizer.transform(query_strings)
print("Query matrix shape:", X_queries.shape)

Query matrix shape: (1109, 50625)


In [8]:
#Retrieval
rankings_vsm = retrieve_documents(
    X_queries,
    X_corpus,
    corpus_ids,
)

In [9]:
#Verification
sample_q_idx = 0

print("Query:")
print(query_strings[sample_q_idx])
print("\nTop Retrieved Documents:\n")
for doc_id, score in rankings_vsm[sample_q_idx]:
    print(f"Doc ID: {doc_id}")
    print(f"Score : {score:.4f}")
    print(f"Title : {corpus[doc_id]['title']}")
    print("-" * 50)

Query:
0dimensional biomaterials lack inductive properties

Top Retrieved Documents:

Doc ID: 10931595
Score : 0.0621
Title : Geometry, epistasis, and developmental patterning.
--------------------------------------------------
Doc ID: 13231899
Score : 0.0600
Title : In situ regulation of DC subsets and T cells mediates tumor regression in mice.
--------------------------------------------------
Doc ID: 42731834
Score : 0.0575
Title : Lack of Absent in Melanoma 2 (AIM2) expression in tumor cells is closely associated with poor survival in colorectal cancer patients.
--------------------------------------------------
Doc ID: 3845894
Score : 0.0560
Title : Computational and Statistical Analyses of Amino Acid Usage and Physico-Chemical Properties of the Twelve Late Embryogenesis Abundant Protein Classes
--------------------------------------------------
Doc ID: 42240424
Score : 0.0553
Title : The effects of prion protein proteolysis and disaggregation on the strain properties of hamster s

### 2. VSM in LSI space

1. Reduce the dimensions of vectors using SVD
2. Perform VSM retrieval in this reduced dimensional space

In [10]:
#Preparing Document Vectors
from sklearn.decomposition import TruncatedSVD

n_components = 300

svd = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

X_corpus_lsi = svd.fit_transform(X_corpus)

print("LSI corpus shape:", X_corpus_lsi.shape)

LSI corpus shape: (5183, 300)


In [11]:
#Preparing Query Vectors
X_queries_lsi = svd.transform(X_queries)

print("LSI query shape:", X_queries_lsi.shape)

LSI query shape: (1109, 300)


In [12]:
#Retrieval of relevant Documents
rankings_lsi = retrieve_documents(
    query_vectors=X_queries_lsi,
    document_vectors=X_corpus_lsi,
    corpus_ids=corpus_ids,
    top_k=10
)

In [13]:
#Verification
sample_q_idx = 0

print("Query:")
print(query_strings[sample_q_idx])
print("\nTop Retrieved Documents:\n")
for doc_id, score in rankings_lsi[sample_q_idx]:
    print(f"Doc ID: {doc_id}")
    print(f"Score : {score:.4f}")
    print(f"Title : {corpus[doc_id]['title']}")
    print("-" * 50)

Query:
0dimensional biomaterials lack inductive properties

Top Retrieved Documents:

Doc ID: 825728
Score : 0.3737
Title : Metastatic colonization requires the repression of the epithelial-mesenchymal transition inducer Prrx1.
--------------------------------------------------
Doc ID: 42240424
Score : 0.3499
Title : The effects of prion protein proteolysis and disaggregation on the strain properties of hamster scrapie.
--------------------------------------------------
Doc ID: 18953920
Score : 0.3401
Title : The Epithelial-Mesenchymal Transition Generates Cells with Properties of Stem Cells
--------------------------------------------------
Doc ID: 1791714
Score : 0.3107
Title : Spatiotemporal regulation of epithelial-mesenchymal transition is essential for squamous cell carcinoma metastasis.
--------------------------------------------------
Doc ID: 1292369
Score : 0.2924
Title : Biochemical Properties of Highly Neuroinvasive Prion Strains
--------------------------------------------

### 3.VSM with Transformer 

The transformer 'all-MiniLM-L6-v2' is used to convert each document or query into **384**-Dimensional Embeddings

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\navan\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\navan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### Prepare Raw Text

In [15]:
corpus_ids = list(corpus.keys())

corpus_texts = [
    doc.get('title', '') + " " + doc.get('text', '')
    for doc in corpus.values()
]

query_ids = list(queries.keys())
query_texts = list(queries.values())

#### Generate Document Embeddings

In [16]:
doc_embeddings = model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(doc_embeddings.shape)

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

(5183, 384)


#### Generate Query Embeddings

In [17]:
query_embeddings = model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(query_embeddings.shape)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

(1109, 384)


#### Ranking the Documents for each query

In [18]:
rankings_embed = retrieve_documents(
    query_vectors=query_embeddings,
    document_vectors=doc_embeddings,
    corpus_ids=corpus_ids,
    top_k=10
)

#### Verification

In [20]:
sample_q_idx = 0

print("Query:")
print(query_strings[sample_q_idx])
print("\nTop Retrieved Documents:\n")
for doc_id, score in rankings_embed[sample_q_idx]:
    print(f"Doc ID: {doc_id}")
    print(f"Score : {score:.4f}")
    print(f"Title : {corpus[doc_id]['title']}")
    print("-" * 50)

Query:
0dimensional biomaterials lack inductive properties

Top Retrieved Documents:

Doc ID: 29638116
Score : 0.3453
Title : Complex Tissue and Disease Modeling using hiPSCs.
--------------------------------------------------
Doc ID: 4346436
Score : 0.3333
Title : Nonlinear Elasticity in Biological Gels
--------------------------------------------------
Doc ID: 17388232
Score : 0.3129
Title : Mechanical regulation of cell function with geometrically modulated elastomeric substrates
--------------------------------------------------
Doc ID: 31715818
Score : 0.2879
Title : New opportunities: the use of nanotechnologies to manipulate and track stem cells.
--------------------------------------------------
Doc ID: 927561
Score : 0.2871
Title : Emergent structures and dynamics of cell colonies by contact inhibition of locomotion
--------------------------------------------------
Doc ID: 3874000
Score : 0.2828
Title : Tissue Mechanics Orchestrate Wnt-Dependent Human Embryonic Stem Cell Diff

## Step 4:- Evaluation

To determine which search engine performs better, we run both predicted top-10 lists against a human-curated answer key (`qrels_all`). We evaluate performance globally using three core metrics averaged across all queries:

1.  **Precision@10:** Measures precision/noise. Out of the 10 documents returned by the engine, what percentage are actually marked as relevant in the answer key? 
2.  **Recall@10:** Measures comprehensiveness/coverage. Out of all the relevant documents that exist for a query in the database, what percentage did our engine manage to catch within its top 10 slots?
3.  **NDCG@10 (Normalized Discounted Cumulative Gain):** Measures rank order quality. It evaluates whether the engine placed the absolute best, highly relevant documents at the very top (Ranks 1, 2, 3) rather than burying them down at Rank 10, penalizing poor placement logarithmically.

### Converting rankings
We stored rankings with the Query indices. I am using already written code for evaluating the models which uses query_ids. So we are converting the Rankings keys from indices to Query_ids as in the data.

In [ ]:
def convert_rankings_to_query_ids(rankings, query_ids):
    converted = {}

    for q_idx, retrieved_docs in rankings.items():
        query_id = query_ids[q_idx]
        converted[query_id] = retrieved_docs

    return converted

rankings_vsm = convert_rankings_to_query_ids(
    rankings_vsm,
    query_ids
)

rankings_lsi = convert_rankings_to_query_ids(
    rankings_lsi,
    query_ids
)

rankings_embed = convert_rankings_to_query_ids(
    rankings_embed,
    query_ids
)

In [25]:
import math

def evaluate_rankings(rankings, qrels, k=10):
    total_queries = 0
    sum_precision = 0.0
    sum_recall = 0.0
    sum_ndcg = 0.0

    for query_id, true_docs in qrels.items():
        if query_id not in rankings:
            continue

        predicted_docs = [doc_id for doc_id, score in rankings[query_id][:k]]
        total_relevant = len(true_docs)

        if total_relevant == 0:
            continue

        total_queries += 1
        relevant_retrieved = 0
        dcg = 0.0

        # 1. Calculate Precision, Recall, and DCG
        for rank_idx, doc_id in enumerate(predicted_docs):
            if doc_id in true_docs and true_docs[doc_id] > 0:
                relevant_retrieved += 1
                relevance_score = true_docs[doc_id]
                
                # DCG Formula: Relevance / log2(Rank + 1)
                # rank_idx is 0-based, so rank_idx + 2 ensures Rank 1 divides by log2(2)
                dcg += relevance_score / math.log2(rank_idx + 2)

        precision_at_k = relevant_retrieved / k
        recall_at_k = relevant_retrieved / total_relevant

        # 2. Calculate Ideal DCG (IDCG)
        ideal_relevance_scores = sorted(true_docs.values(), reverse=True)
        idcg = 0.0
        for rank_idx, rel in enumerate(ideal_relevance_scores[:k]):
            idcg += rel / math.log2(rank_idx + 2)

        # NDCG is the ratio of what we got vs. what was ideally possible
        ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

        sum_precision += precision_at_k
        sum_recall += recall_at_k
        sum_ndcg += ndcg_at_k

    return (sum_precision / total_queries), (sum_recall / total_queries), (sum_ndcg / total_queries)

print("Evaluating systems against ground truth...")

# Grade the Models
vsm_p, vsm_r, vsm_n = evaluate_rankings(rankings_vsm, qrels_all, k=10)
lsi_p, lsi_r, lsi_n = evaluate_rankings(rankings_lsi, qrels_all, k=10)
embed_p, embed_r, embed_n = evaluate_rankings(rankings_embed, qrels_all, k=10)

print("\n" + "="*50)
print(f"🏆 FINAL LEADERBOARD (Top 10 Results)")
print("="*50)
print(f"{'Metric':<15} | {'Normal VSM':<20} | {'VSM in LSI space':<20} | {'VSM with Embed':<20}")
print("-" * 50)
print(f"{'Precision@10':<15} | {vsm_p:<15.4f} | {lsi_p:<15.4f} | {embed_p:<15.4f}")
print(f"{'Recall@10':<15} | {vsm_r:<15.4f} | {lsi_r:<15.4f} | {embed_r:<15.4f}")
print(f"{'NDCG@10':<15} | {vsm_n:<15.4f} | {lsi_n:<15.4f} | {embed_n:<15.4f}")
print("="*50)

Evaluating systems against ground truth...

🏆 FINAL LEADERBOARD (Top 10 Results)
Metric          | Normal VSM           | VSM in LSI space     | VSM with Embed      
--------------------------------------------------
Precision@10    | 0.0814          | 0.0610          | 0.0890         
Recall@10       | 0.7286          | 0.5358          | 0.7850         
NDCG@10         | 0.5746          | 0.3944          | 0.6561         


### Conclusion:
Among the three approaches, VSM with Embeddings achieved the best performance, obtaining the highest Precision@10 (0.0890), Recall@10 (0.7850), and NDCG@10 (0.6561). The Normal TF-IDF VSM performed moderately well, while VSM in LSI space showed the lowest scores across all metrics. These results indicate that semantic embeddings capture document-query relevance more effectively than traditional TF-IDF and LSI-based representations on the SciFact dataset.